### src/config.py

In [1]:
from pathlib import Path

#BASE_DIR = Path(__file__).resolve().parent.parent
BASE_DIR = Path("/Users/andresluna/mlprojects/ml_churn")

DATA_DIR = BASE_DIR / "data"
RAW_DATA_PATH = DATA_DIR / "raw" / "telco_churn.csv"

MODEL_DIR = BASE_DIR / "models"
MODEL_PATH = MODEL_DIR / "v1"

LOG_DIR = BASE_DIR / "logs"

CONFIG = {
    "target_column": "Churn",
    "target_map" : {'Yes': 1, 'No': 0},
    "test_size": 0.2,
    "random_state": 42,
    "model_type": "logistic_regression"  # or "random_forest"
}

MODELS_FAMILY = {"logistic_regression" : "linear", "random_forest" : "tree", "gradient_boosting" : "tree"}

### src/logger.py

In [2]:
import logging
from pathlib import Path
#from src.config import LOG_DIR

def setup_logger(name: str):
    LOG_DIR.mkdir(exist_ok=True)

    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)

    if not logger.handlers:
        formatter = logging.Formatter(
            "%(asctime)s | %(name)s | %(levelname)s | %(message)s"
        )

        file_handler = logging.FileHandler(LOG_DIR / f"{name}.log")
        file_handler.setFormatter(formatter)

        stream_handler = logging.StreamHandler()
        stream_handler.setFormatter(formatter)

        logger.addHandler(file_handler)
        logger.addHandler(stream_handler)

    return logger

### src/data_loader.py

In [3]:
import pandas as pd
#from src.config import RAW_DATA_PATH

def load_raw_data(path=RAW_DATA_PATH):
    df = pd.read_csv(path)
    return df

### src/preprocessing.py

In [4]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

### CLEANING DATA

class DataCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = X.copy()

        # Numeric conversion
        X['TotalCharges'] = pd.to_numeric(X['TotalCharges'], errors='coerce')

        # Binary mappings (NO Churn here!)
        binary_map = {
            'gender': {'Female': 1, 'Male': 0},
            'Partner': {'Yes': 1, 'No': 0},
            'Dependents': {'Yes': 1, 'No': 0},
            'PhoneService': {'Yes': 1, 'No': 0},
            'PaperlessBilling': {'Yes': 1, 'No': 0},
        }

        for col, mapping in binary_map.items():
            if col in X:
                X[col] = X[col].map(mapping)

        # MultipleLines
        if 'MultipleLines' in X:
            X['MultipleLines'] = (X['MultipleLines'] == 'Yes').astype(int)

        # Internet-dependent features
        internet_features = [
            'OnlineSecurity','OnlineBackup','DeviceProtection',
            'TechSupport','StreamingTV','StreamingMovies'
        ]

        for col in internet_features:
            if col in X:
                X[col] = (X[col] == 'Yes').astype(int)

        return X
    


### FEATURE ENGINEERING

class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.median_charge_ = X['MonthlyCharges'].median()
        return self

    def transform(self, X):
        X = X.copy()

        # Groups
        X['tenure_group'] = pd.cut(
            X['tenure'],
            bins=[0, 12, 24, 48, 72],
            labels=['0-1yr', '1-2yr', '2-4yr', '4-6yr']
        )

        X['avg_revenue'] = X['TotalCharges'] / (X['tenure'] + 1)

        return X
    
class FlagBuilder(BaseEstimator, TransformerMixin):
    def __init__(self, flag_configs):
        self.flag_configs = flag_configs

    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = X.copy()
        for new_col, func in self.flag_configs.items():
            X[new_col] = func(X).astype(int)
        return X


class HighMonthlyFlag(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.median_ = X['MonthlyCharges'].median()
        return self

    def transform(self, X):
        X = X.copy()
        X['high_monthly_flag'] = (X['MonthlyCharges'] > self.median_).astype(int)
        return X
    

class ServiceCounter(BaseEstimator, TransformerMixin):
    def __init__(self, service_cols):
        self.service_cols = service_cols

    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = X.copy()
        X['num_services'] = (X[self.service_cols] == 'Yes').sum(axis=1)
        return X
    

class InteractionBuilder(BaseEstimator, TransformerMixin):
    def __init__(self, interactions):
        self.interactions = interactions

    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = X.copy()
        for new_col, (c1, c2) in self.interactions.items():
            X[new_col] = X[c1] * X[c2]
        return X


def is_new_customer(X):
    return X['tenure'] <= 12
def is_long_term(X):
    return X['Contract'].isin(['One year', 'Two year'])
def auto_pay_flag(X):
    return X['PaymentMethod'].isin(['Bank transfer (automatic)', 'Credit card (automatic)'])
def family_flag(X):
    return (X['Partner'] == 1) | (X['Dependents'] == 1)  # assuming binary mapping already applied
def fiber_flag(X):
    return X['InternetService'] == 'Fiber optic'
def electronic_check_flag(X):
    return X['PaymentMethod'] == 'Electronic check'

class FlagBuilder(BaseEstimator, TransformerMixin):
    def __init__(self, flag_funcs):
        """
        flag_funcs: dict, keys are new column names, values are functions X -> series
        """
        self.flag_funcs = flag_funcs

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col, func in self.flag_funcs.items():
            X[col] = func(X).astype(int)
        return X
    
flag_builder = FlagBuilder({
    'is_new_customer': is_new_customer,
    'is_long_term': is_long_term,
    'auto_pay_flag': auto_pay_flag,
    'family_flag': family_flag,
    'fiber_flag': fiber_flag,
    'electronic_check_flag': electronic_check_flag
})

feature_pipeline = Pipeline([
    ('basic', FeatureEngineer()),
    ('flags', flag_builder),
    ('high_monthly', HighMonthlyFlag()),
    ('services', ServiceCounter([
        'OnlineSecurity','OnlineBackup','DeviceProtection',
        'TechSupport','StreamingTV','StreamingMovies'])),
    ('interactions', InteractionBuilder({
        'new_echeck_interaction': ('is_new_customer', 'electronic_check_flag'),
        'fiber_highcharge_interaction': ('fiber_flag', 'high_monthly_flag'),
        'loyal_engaged_interaction': ('is_long_term', 'num_services'),}))])


## REAUSABLE PREPROCESSING COMPONENTS

median_imputer = SimpleImputer(strategy='median')
most_frequent_imputer = SimpleImputer(strategy='most_frequent')
missing_cat_imputer = SimpleImputer(strategy='constant', fill_value='Missing')

scaler = StandardScaler()

ohe_drop_first = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
ohe_all = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

## FEATURES

numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'avg_revenue']
categorical_features = ['Contract', 'PaymentMethod', 'InternetService', 'tenure_group']
engineered_features = [
    'is_new_customer', 
    'is_long_term', 
    'auto_pay_flag', 
    'num_services',
    'high_monthly_flag', 
    'family_flag', 
    'fiber_flag', 
    'electronic_check_flag',
    'new_echeck_interaction', 
    'fiber_highcharge_interaction', 
    'loyal_engaged_interaction'
]


### LINEAR AND TREE PREPROCESSING


def get_preprocessor(numeric_features, categorical_features, engineered_features):
    # Linear model preprocessing
    num_pipeline_linear = Pipeline([
        ('imputer', median_imputer),
        ('scaler', scaler)])
    cat_pipeline_linear = Pipeline([
        ('imputer', missing_cat_imputer),
        ('encoder', ohe_drop_first)])
    linear_preprocessor = ColumnTransformer([
        ('num', num_pipeline_linear, numeric_features + engineered_features),
        ('cat', cat_pipeline_linear, categorical_features)])

    # Tree-based preprocessing
    num_pipeline_tree = Pipeline([
        ('imputer', median_imputer)])
    cat_pipeline_tree = Pipeline([
        ('imputer', most_frequent_imputer),
        ('encoder', ohe_all)])
    tree_preprocessor = ColumnTransformer([
        ('num', num_pipeline_tree, numeric_features + engineered_features),
        ('cat', cat_pipeline_tree, categorical_features)])

    ## PREPROCESSOR PIPELINES
    full_pipeline_linear = Pipeline([
        ('cleaning', DataCleaner()),
        ('feature_engineering', feature_pipeline),
        ('preprocessing', linear_preprocessor)])
    full_pipeline_tree = Pipeline([
        ('cleaning', DataCleaner()),
        ('feature_engineering', feature_pipeline),
        ('preprocessing', tree_preprocessor)])
    return full_pipeline_linear, full_pipeline_tree

full_pipeline_linear, full_pipeline_tree = get_preprocessor(numeric_features, categorical_features,engineered_features)

### src/train_model.py

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

def train_model(X, y, model_type):

    if model_type == "logistic_regression":
        # Instantiate baseline logistic regression
        model = LogisticRegression(random_state=42, max_iter=1000)

    elif model_type == "random_forest":
        # Instantiate baseline random forest
        model = RandomForestClassifier(random_state=42, n_estimators=100)

    elif model_type == "gradient_boosting":
        # Instantiate baseline gradient boosting
        model = GradientBoostingClassifier(random_state=42, n_estimators=100, learning_rate=0.1)

    model.fit(X,y)
    return model

### src/evaluate.py

In [6]:

from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred)
    }

    return metrics 

### scripts/train.py

In [7]:
from sklearn.model_selection import train_test_split
import joblib
from datetime import datetime
from pathlib import Path
import pandas as pd

timestamp = datetime.now().strftime("%y%m%d_%H%M%S")
logger = setup_logger(f"train_pipeline_{timestamp}")

# Start logger
logger.info("Starting training pipeline")

MODEL_PATH.mkdir(parents=True, exist_ok=True)

# Load data
df_raw = load_raw_data()

logger.info(f"Loaded data with shape {df_raw.shape}")

# Split features and target
X_raw = df_raw.drop(columns=CONFIG['target_column'])
y = df_raw[CONFIG['target_column']].map(CONFIG['target_map'])

logger.info(f"Split features and target {CONFIG['target_column']}")

# Split train and test sets
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=CONFIG['test_size'], stratify=y, random_state=CONFIG['random_state']
)

logger.info(f"Split train and test set of size {CONFIG['test_size']}")

# Preprocessing
if MODELS_FAMILY[CONFIG['model_type']] == "linear":
    preprocessor = full_pipeline_linear
elif MODELS_FAMILY[CONFIG['model_type']] == "tree":
    preprocessor = full_pipeline_tree

preprocessor.fit(X_train_raw, y_train)
X_train = preprocessor.transform(X_train_raw)
X_test  = preprocessor.transform(X_test_raw)

logger.info(f"Finished preprocessing for a {MODELS_FAMILY[CONFIG['model_type']]}-type model")

# Train model
model = train_model(X_train, y_train, CONFIG['model_type'])
logger.info(f"Completed training a {CONFIG['model_type']} model")

# Evaluate
metrics = evaluate_model(model, X_test, y_test)
logger.info(f"Metrics: {metrics}")

# Save artifacts
joblib.dump(model, MODEL_PATH / "model.pkl")
#joblib.dump(preprocessor, MODEL_PATH / "preprocessor.pkl")

logger.info(f"Saved model and preprocessor to {MODEL_PATH}")
logger.info("Training pipeline finished successfully")


2026-03-04 15:10:16,937 | train_pipeline_260304_151016 | INFO | Starting training pipeline
2026-03-04 15:10:16,955 | train_pipeline_260304_151016 | INFO | Loaded data with shape (7043, 21)
2026-03-04 15:10:16,958 | train_pipeline_260304_151016 | INFO | Split features and target Churn
2026-03-04 15:10:16,963 | train_pipeline_260304_151016 | INFO | Split train and test set of size 0.2
2026-03-04 15:10:17,029 | train_pipeline_260304_151016 | INFO | Finished preprocessing for a linear-type model
2026-03-04 15:10:17,043 | train_pipeline_260304_151016 | INFO | Completed training a logistic_regression model
2026-03-04 15:10:17,048 | train_pipeline_260304_151016 | INFO | Metrics: {'accuracy': 0.7927608232789212, 'roc_auc': 0.8413482652613088, 'precision': 0.6453900709219859, 'recall': 0.48663101604278075, 'f1_score': 0.5548780487804879}
2026-03-04 15:10:17,050 | train_pipeline_260304_151016 | INFO | Saved model and preprocessor to /Users/andresluna/mlprojects/ml_churn/models/v1
2026-03-04 15:1